In [0]:
%python
# restaurant_id STRING PRIMARY KEY,
# restaurant_name STRING,
# city STRING,
# total_reviews BIGINT,
# avg_rating DECIMAL(3,2)
# rating_5_count BIGINT,
# rating_4_count BIGINT,
# rating_3_count BIGINT,
# rating_2_count BIGINT,
# rating_1_count BIGINT,
# sentiment_positive_count BIGINT,
# sentiment_neutral_count BIGINT,
# sentiment_negative_count BIGINT

In [0]:
%python
from pyspark.sql.functions import *

In [0]:
%python
df_review_stats = (
    spark.read.table("`restaurantopcat`.`02_silver`.`fact_reviews`")
    .groupBy("restaurant_id")
   .agg(
        countDistinct("review_id").alias("total_reviews"),
        round(avg("rating"), 2).alias("avg_rating"),
        sum(when(col("rating") == 5, 1).otherwise(0)).alias("rating_5_count"),
        sum(when(col("rating") == 4, 1).otherwise(0)).alias("rating_4_count"),
        sum(when(col("rating") == 3, 1).otherwise(0)).alias("rating_3_count"),
        sum(when(col("rating") == 2, 1).otherwise(0)).alias("rating_2_count"),
        sum(when(col("rating") == 1, 1).otherwise(0)).alias("rating_1_count"),

        sum(when(col("sentiment") == "positive", 1).otherwise(0)).alias("sentiment_positive_count"),
        sum(when(col("sentiment") == "neutral", 1).otherwise(0)).alias("sentiment_neutral_count"),
        sum(when(col("sentiment") == "negative", 1).otherwise(0)).alias("sentiment_negative_count")
    )
)

display(df_review_stats)


restaurant_id,total_reviews,avg_rating,rating_5_count,rating_4_count,rating_3_count,rating_2_count,rating_1_count,sentiment_positive_count,sentiment_neutral_count,sentiment_negative_count
REST-AUH-002,25,4.28,14,7,2,1,1,21,2,2
REST-DXB-002,14,3.86,2,9,2,1,0,11,2,1
REST-SHJ-001,11,4.45,7,3,0,1,0,10,0,1
REST-DXB-001,14,4.21,7,4,2,1,0,11,1,2
REST-AUH-001,14,4.07,6,5,1,2,0,11,1,2


In [0]:
%python
df_restaurant = spark.table("`restaurantopcat`.`02_silver`.`dim_resutrants`")

df_restaurant_reviews = (
    df_restaurant.join(df_review_stats, on="restaurant_id", how="left")
    .select(
        "restaurant_id",
        col("name").alias("restaurant_name"),
        "city",

        coalesce(col("total_reviews"), lit(0)).alias("total_reviews"),
        coalesce(col("avg_rating"), lit(0)).alias("avg_rating"),
        coalesce(col("rating_5_count"), lit(0)).alias("rating_5_count"),
        coalesce(col("rating_4_count"), lit(0)).alias("rating_4_count"),
        coalesce(col("rating_3_count"), lit(0)).alias("rating_3_count"),
        coalesce(col("rating_2_count"), lit(0)).alias("rating_2_count"),
        coalesce(col("rating_1_count"), lit(0)).alias("rating_1_count"),

        coalesce(col("sentiment_positive_count"), lit(0)).alias("sentiment_positive_count"),
        coalesce(col("sentiment_neutral_count"), lit(0)).alias("sentiment_neutral_count"),
        coalesce(col("sentiment_negative_count"), lit(0)).alias("sentiment_negative_count"),
    )
)

display(df_restaurant_reviews)

restaurant_id,restaurant_name,city,total_reviews,avg_rating,rating_5_count,rating_4_count,rating_3_count,rating_2_count,rating_1_count,sentiment_positive_count,sentiment_neutral_count,sentiment_negative_count
REST-AUH-001,Spice Route Downtown,Abu Dhabi,14,4.07,6,5,1,2,0,11,1,2
REST-AUH-002,Spice Route Al Wahda,Abu Dhabi,25,4.28,14,7,2,1,1,21,2,2
REST-DXB-001,Spice Route Marina,Dubai,14,4.21,7,4,2,1,0,11,1,2
REST-DXB-002,Spice Route Mall of Emirates,Dubai,14,3.86,2,9,2,1,0,11,2,1
REST-SHJ-001,Spice Route City Centre,Sharjah,11,4.45,7,3,0,1,0,10,0,1
